# Validadores *after*

Como vimos nos slides, um validador customizado é essencialmente uma **função de transformação**: recebe uma entrada e precisa devolver o
valor que está sendo desserializado.

Também podemos aproveitar para fazer verificações e levantar exceção quando for o caso.

Ou seja, validadores customizados não são só validadores: também transformam os dados desserializados. Isso é importante, porque às
vezes são usados só para validar, às vezes só para transformar, e às vezes para os dois.

Nesta aula começamos definindo validadores customizados com um **decorador**:

In [1]:
from pydantic import BaseModel, Field, field_validator, ValidationError

No primeiro exemplo, faremos um validador que verifica se um inteiro é par.

Poderíamos (e provavelmente deveríamos) usar a restrição `multiple_of` do `Field`, mas o exemplo serve para ilustrar.

O comportamento padrão dos validadores definidos com decorador é ser **AFTER**.

Ou seja: o valor que chega ao validador já foi processado pelo Pydantic de acordo com o tipo e as demais restrições do campo.

In [2]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

Definimos um campo inteiro, maior que `0` e menor que `10`.

Assim como está, o Pydantic roda sua validação e coerção interna nos dados de entrada:

In [3]:
print(f'Model(number="4"): {Model(number="4")!r}')

Model(number="4"): Model(number=4)


A string `"4"` virou inteiro. Além disso, o Pydantic garante que o número está dentro dos limites.

In [4]:
try:
    Model(number=12)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be less than 10 [type=less_than, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than


Agora adicionamos um validador *after* ao campo.

Validadores customizados são métodos **de classe** — ficam ligados à classe, não à instância — e devemos definir a assinatura de acordo,
além de marcar a função como `classmethod` (não é obrigatório, mas é melhor).

In [5]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        return value  # validadores customizados precisam devolver um valor

Testando com uma entrada válida:

In [6]:
print(f"Model(number=3): {Model(number=3)!r}")

Running custom validator
value=3, type(value)=<class 'int'>
Model(number=3): Model(number=3)


O validador foi chamado, e o argumento `value` recebido é um inteiro. Você poderia pensar que é porque passamos um inteiro ao criar a
instância, mas veja:

In [7]:
print(f'Model(number="3"): {Model(number="3")!r}')

Running custom validator
value=3, type(value)=<class 'int'>
Model(number="3"): Model(number=3)


O validador continuou recebendo um inteiro.

E se passarmos um valor que não satisfaz as restrições do Pydantic:

In [8]:
try:
    Model(number=12)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
number
  Input should be less than 10 [type=less_than, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than


O validador customizado **NÃO** foi chamado.

É assim que funciona um validador **after**: é chamado depois que os validadores/desserializadores do próprio Pydantic terminam (com sucesso).

Como definimos `number` como inteiro entre 0 e 10, quando o nosso validador roda, temos **garantia** de que `value` satisfaz o tipo e as
restrições.

O validador vira apenas uma verificação extra, além das validações que o Pydantic já fez.

Completando o validador.

Para indicar erro de validação no validador customizado, basta levantar `ValueError` com uma mensagem adequada. Primeiro, veja que uma
**subclasse** de `ValueError` também funciona:

In [9]:
class NotCustomValidated(ValueError):
    pass

In [10]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        if value % 2 == 0:
            # número par: devolve
            return value
        raise NotCustomValidated("value must be even")

In [11]:
try:
    Model(number=3)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

Running custom validator
value=3, type(value)=<class 'int'>
ValidationError: 1 validation error for Model
number
  Value error, value must be even [type=value_error, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [12]:
try:
    Model(number=3)
except Exception as ex:
    print(f"{type(ex).__name__}: {ex}")

Running custom validator
value=3, type(value)=<class 'int'>
ValidationError: 1 validation error for Model
number
  Value error, value must be even [type=value_error, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


Mesmo com a exceção customizada, o que chega ao chamador é um `ValidationError`. Agora com `ValueError` puro:

In [13]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        if value % 2 == 0:
            # número par: devolve
            return value
        raise ValueError("value must be even")

Testando:

In [14]:
print(f"Model(number=4): {Model(number=4)!r}")

Running custom validator
value=4, type(value)=<class 'int'>
Model(number=4): Model(number=4)


In [15]:
try:
    Model(number=3)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

Running custom validator
value=3, type(value)=<class 'int'>
ValidationError: 1 validation error for Model
number
  Value error, value must be even [type=value_error, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


A exceção levantada foi `ValidationError`, não `ValueError`. O Pydantic captura qualquer `ValueError` do validador e levanta um
`ValidationError` no lugar.

Outros erros, como `TypeError`, **não** são tratados pelo Pydantic e sobem sem alteração:

In [16]:
class Model(BaseModel):
    number: int = Field(gt=0, lt=10)

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        print("Running custom validator")
        print(f"{value=}, {type(value)=}")
        if value % 2 == 0:
            # número par: devolve
            return value
        raise TypeError("value must be even")

In [17]:
try:
    Model(number=3)
except Exception as ex:
    print(f"{type(ex)=}, {ex}")

Running custom validator
value=3, type(value)=<class 'int'>
type(ex)=<class 'TypeError'>, value must be even


O `TypeError` subiu como ele mesmo, não como erro de validação. Então, para manter a consistência (o Pydantic levanta `ValidationError`),
levantamos `ValueError`.

A documentação do Pydantic diz claramente para **não** levantarmos `ValidationError` nós mesmos — levantamos `ValueError` e o Pydantic
cuida do resto.

> 💡 **Complemento (slide 5):** `assert` também vira `ValidationError`... **exceto** quando o Python roda com `-O`, que remove os asserts.
> A validação simplesmente desaparece, sem erro. Por isso: nunca valide com `assert`.

In [18]:
class AssertModel(BaseModel):
    number: int

    @field_validator("number")
    @classmethod
    def validate_even(cls, value):
        assert value % 2 == 0, "value must be even"
        return value

try:
    AssertModel(number=3)
except ValidationError as ex:
    print(f'ex.errors()[0]["type"]: {ex.errors()[0]["type"]}')
    print(f'ex.errors()[0]["msg"]: {ex.errors()[0]["msg"]}')

ex.errors()[0]["type"]: assertion_error
ex.errors()[0]["msg"]: Assertion failed, value must be even


Como mencionado, o validador customizado também é um bom lugar para **modificar** o valor desserializado, se necessário.

Exemplo simples: queremos um número par e, se o número não for par, trocamos pelo próximo par.

(Exemplo meio bobo — veremos algo mais realista em seguida.)

In [19]:
class Model(BaseModel):
    even: int

    @field_validator("even")
    @classmethod
    def make_even(cls, value: int) -> int:
        if value % 2 == 1:
            return value + 1
        return value

O validador não precisa verificar se `value` é inteiro — sendo *after*, isso já está garantido.

In [20]:
print(f'Model(even="3"): {Model(even="3")!r}')

Model(even="3"): Model(even=4)


Um exemplo mais realista de transformação.

Suponha um campo `datetime` que queremos sempre como datetime UTC *aware*.

Ou seja: se o datetime de entrada for *naive*, assumimos que já está em UTC e só o tornamos *aware*; se for *aware*, convertemos para UTC.

É parecido com o exemplo dos serializadores customizados. Mas, em vez de transformar na **serialização**, transformamos na **desserialização**.

In [21]:
from datetime import datetime
import pytz

def make_utc(dt: datetime) -> datetime:
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt

É a função que usamos antes.

Agora um modelo que garante datetimes sempre armazenados como UTC *aware*.

In [22]:
class Model(BaseModel):
    dt: datetime

    @field_validator("dt")
    @classmethod
    def make_utc(cls, dt: datetime) -> datetime:
        if dt.tzinfo is None:
            dt = pytz.utc.localize(dt)
        else:
            dt = dt.astimezone(pytz.utc)
        return dt

Testando:

In [23]:
print(f'Model(dt="2020-01-01T03:00:00"): {Model(dt="2020-01-01T03:00:00")!r}')

Model(dt="2020-01-01T03:00:00"): Model(dt=datetime.datetime(2020, 1, 1, 3, 0, tzinfo=<UTC>))


E com um datetime *aware*:

In [24]:
eastern = pytz.timezone('US/Eastern')
dt = eastern.localize(datetime(2020, 1, 1, 3, 0, 0))

In [25]:
print(f"Model(dt=dt): {Model(dt=dt)!r}")

Model(dt=dt): Model(dt=datetime.datetime(2020, 1, 1, 8, 0, tzinfo=<UTC>))


Podemos ter vários validadores no mesmo campo.

Como vimos nos slides, validadores *after* executam na ordem de definição, de cima para baixo.

In [26]:
class Model(BaseModel):
    number: int

    @field_validator("number")
    @classmethod
    def add_1(cls, value: int):
        print(f"running add_1({value}) -> {value + 1}")
        return value + 1

    @field_validator("number")
    @classmethod
    def add_2(cls, value: int):
        print(f"running add_2({value}) -> {value + 2}")
        return value + 2

    @field_validator("number")
    @classmethod
    def add_3(cls, value: int):
        print(f"running add_3({value}) -> {value + 3}")
        return value + 3

In [27]:
print(f"Model(number=1): {Model(number=1)!r}")

running add_1(1) -> 2
running add_2(2) -> 4
running add_3(4) -> 7
Model(number=1): Model(number=7)


Os três validadores rodaram na ordem de definição, de cima para baixo.

Também dá para aplicar o mesmo validador a vários campos.

Suponha uma classe com alguns floats que queremos sempre arredondados para 2 casas decimais.

Fácil com um validador customizado:

In [28]:
class Model(BaseModel):
    unit_cost: float
    unit_price: float

    @field_validator("unit_cost", "unit_price")
    @classmethod
    def round_2(cls, value: float) -> float:
        return round(value, 2)

In [29]:
print(f"Model(unit_cost=2.12345, unit_price=5.9876): {Model(unit_cost=2.12345, unit_price=5.9876)!r}")

Model(unit_cost=2.12345, unit_price=5.9876): Model(unit_cost=2.12, unit_price=5.99)


Para aplicar o validador a **todos** os campos da classe, usamos o indicador especial `"*"` no lugar dos nomes:

In [30]:
class Model(BaseModel):
    unit_cost: float
    unit_price: float

    @field_validator("*")
    @classmethod
    def round_2(cls, value: float) -> float:
        return round(value, 2)

In [31]:
print(f"Model(unit_cost=2.12345, unit_price=5.9876): {Model(unit_cost=2.12345, unit_price=5.9876)!r}")

Model(unit_cost=2.12345, unit_price=5.9876): Model(unit_cost=2.12, unit_price=5.99)
